# 08 — Production: Multimodal (Optional)

**Stage 8 of the workshop.** An artist agent generates images, a critic agent evaluates them — currently cut from the live agenda since local Ollama models have no vision.

## Problem

Getting from "works on my laptop" to something a team can rely on — different model provider, observability, deployment target. This script specifically stresses the model-provider swap: the critic agent's job only makes sense with a vision-capable model, and the local default doesn't have one.

## Concept

The agent logic doesn't fundamentally change when the model provider changes — that's the architectural point paying off here. Two `Agent()` instances share the exact same `text_model` in this cut-down version; the intended design swaps in a vision-capable model (`vision_model`, commented out) for the critic without touching the critic's tools or prompt structure at all.

```
                 Agent
                   │
                   ▼
            Model Interface
                   │
      ┌────────────┼────────────┐
      │            │            │
   Ollama       Bedrock       OpenAI
      │            │            │
     LLM          LLM          LLM
```

## Architecture

```
artist (Agent + generate_image, text_model)
     │  "Generate 3 images of a dog"
     ▼
  tool call: generate_image (x3, varied prompts)
     │
     ▼
  output: comma-separated list of image filesystem paths
     │
     ▼
critic (Agent + image_reader, text_model — should be vision_model)
     │  reads each path, describes it, picks the best
     ▼
  "FINAL DECISION: <path>"
```

## Step 1 — Model setup

Both agents use the same `text_model` here. The commented-out `vision_model` line shows the intended production swap once a Bedrock vision model (or local vision model) is available.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent
from strands_tools import generate_image, image_reader

text_model = get_model()
# vision_model = BedrockModel(model_id="qwen.qwen3-vl-235b-a22b", region_name="ap-south-1")


## Step 2 — Define the artist and critic agents

In [ ]:
artist = Agent(
    model=text_model,
    tools=[generate_image],
    system_prompt=(
        "You will be instructed to generate a number of images of a given subject. "
        "Vary the prompt for each generated image to create a variety of options. "
        "Your final output must contain ONLY a comma-separated list of the "
        "filesystem paths of generated images."
    ),
)

critic = Agent(
    model=text_model,  # swap to vision_model once available locally
    tools=[image_reader],
    system_prompt=(
        "You will be provided with a list of filesystem paths, each containing an "
        "image. Describe each image, and then choose which one is best. "
        "Your final line of output must be: FINAL DECISION: <path to final decision image>"
    ),
)


## Step 3 — Run it

In [ ]:
result = artist("Generate 3 images of a dog")
print(critic(str(result)))


## Failure mode to know about

Local Ollama models used here have no vision. Don't demo unless you've pulled a vision model and re-tested — currently cut from the agenda entirely, keep it that way unless fixed beforehand. `qwen2.5:7b` (the Ollama fallback) is text-only, and `generate_image` needs an image-gen backend Ollama doesn't provide either — this only fully works when Bedrock is reachable with model access granted. See `MODEL_PRICING.md` in this folder for the recommended vision model to swap in for the critic agent.